In [ ]:
import numpy as np
import pandas as pd

'''
This notebook starts by creating three different dataframes based on cvs files and removing null or NaN values for clean data analysis
'''
sales_df = pd.DataFrame(pd.read_csv('./csvs/sales.csv')).dropna()
inventories_df = pd.DataFrame(pd.read_csv('csvs/inventories.csv')).dropna()
satisfaction_df = pd.DataFrame(pd.read_csv('./csvs/satisfaction.csv')).dropna()

'''
Data analysis based on the sales dataframe
'''

total_sales_by_prod_store = sales_df.groupby(['Producto', 'ID_Tienda'])['Cantidad_Vendida'].sum()
print(f"El valor total de las ventas por producto es: \n{total_sales_by_prod_store}")

sales_df['Ingreso'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']
income_by_store = sales_df.groupby(['Producto', 'ID_Tienda'])['Ingreso'].sum()
total_sales_by_store = sales_df.groupby('ID_Tienda')['Ingreso'].sum()
print(f"Resumen de ventas: \n{sales_df.describe()}")
print(f"Los ingresos totales por tienda son: \n{income_by_store}")
print(f"Ventas por tienda: \n{total_sales_by_store}")


'''
Data analysis based on the inventories dataframe
'''

inventories_df['Rotación'] = sales_df['Cantidad_Vendida'] / inventories_df['Stock_Disponible']
inventories_rotation_by_store = inventories_df.groupby(['Producto', 'ID_Tienda'])['Rotación'].sum()

print(f"La rotación del inventario según tienda es: \n{inventories_rotation_by_store}")
low_stock_filter = inventories_df['Rotación'] * 100 < 10
print(f"Estas son las tiendas que tienen el nivel de inventario critico: \n{inventories_df[low_stock_filter]}")

'''
Data analysis based on client satisfaction
'''
satisfaction_by_store = sales_df.merge(satisfaction_df, how='left', on='ID_Tienda')[
    ['ID_Tienda', 'Satisfacción_Promedio']].drop_duplicates()
low_satisfaction = satisfaction_by_store[satisfaction_by_store['Satisfacción_Promedio'] < 60]
print(f"Tiendas con baja satisfacción: \n{low_satisfaction}")

if len(low_satisfaction) > 0:
    low_sat_ids = low_satisfaction['ID_Tienda'].unique()
    low_sat_sales = sales_df[sales_df['ID_Tienda'].isin(low_sat_ids)].groupby('ID_Tienda')['Cantidad_Vendida'].sum()

    print("Recomendaciones:")
    for store_id in low_sat_ids:
        print(f"- Tienda {store_id}: Mejorar atención al cliente.")

else:
    print("No hay tiendas con satisfacción crítica (< 60%)")

# Create Total_Ventas column as required
sales_df['Total_Ventas'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']

# Merge sales with satisfaction data for quantitative analysis
sales_satisfaction = sales_df.merge(satisfaction_df, how='left', on='ID_Tienda')

# Calculate correlation between sales income and satisfaction
correlation = sales_satisfaction['Ingreso'].corr(sales_satisfaction['Satisfacción_Promedio'])
print(f"Correlación entre ingresos y satisfacción: {correlation:.4f}")

# Sales summary by satisfaction level
sales_by_satisfaction = sales_satisfaction.groupby('Satisfacción_Promedio').agg({
    'Total_Ventas': ['sum', 'mean', 'count'],
    'ID_Tienda': 'nunique'
}).round(2)
print(f"Resumen de ventas por nivel de satisfacción:\n{sales_by_satisfaction}")

# Low satisfaction stores - detailed sales analysis
low_sat_stores = sales_satisfaction[sales_satisfaction['Satisfacción_Promedio'] < 60]
low_sat_summary = low_sat_stores.groupby('ID_Tienda')['Total_Ventas'].agg(['sum', 'mean', 'count'])
print(f"Análisis de ventas en tiendas con baja satisfacción:\n{low_sat_summary}")
'''
Numpy operations
'''
np.random.seed(42)
total_sales = sales_df['Total_Ventas'].to_numpy()
median_total_sales = np.median(total_sales)
std_total_sales = np.std(total_sales)
proyectes_sales = np.random.normal(loc=median_total_sales, scale=std_total_sales, size=100)

print(f"Mediana ventas totales: {median_total_sales}")
print(f"Desviación estándar ventas totales: {std_total_sales}")
print(f"Proyecciones de ventas futuras: \n{proyectes_sales}")